# Chinese MNIST – PCA + k-NN Experiment

In this notebook I used the Chinese MNIST CSV dataset as a second playground for k‑NN and PCA, this time on larger and more varied digit images.

## 1. Dataset

- **Source**: Chinese MNIST (CSV version).
- **Shape**: `15000 × 4098`
  - `pixel_0` … `pixel_4095` → 4096 grayscale pixels.
  - `label` → 15 numeric classes (0–14), one per Chinese digit/character.
  - `character` → the corresponding Chinese character (e.g. 九).
- Each image is **64×64** pixels (√4096 = 64).

## 2. Baseline: k-NN on Raw Pixels

- Features: all 4096 pixel columns (no preprocessing).
- Model: `KNeighborsClassifier(n_neighbors=5)`.
- Split: 80% train / 20% test, stratified by label.

**Result**

- Test accuracy ≈ **0.43**.

**Interpretation**

- Much better than random guessing (1/15 ≈ 0.067), but still low.
- In 4096‑dimensional pixel space, Euclidean distance is noisy → nearest neighbours are not very reliable.
- This is a clear example of the **curse of dimensionality**.

## 3. PCA + k-NN (Fixed PCA Dimension)

- Applied **PCA with 50 components** to `X`.
- Trained k‑NN (k=5) on the 50‑dimensional PCA features.

**Result**

- Test accuracy ≈ **0.709**.

**Interpretation**

- Compressing 4096 pixels into 50 principal components produced a **“stroke space”** where similar digits are closer.
- PCA removed a lot of pixel‑level noise but kept the main structure, so k‑NN became much stronger.

## 4. PCA + k-NN with Cross-Validation (Pipeline)

- Built a **Pipeline**: `PCA → k-NN`.
- Tuned hyperparameters with **GridSearchCV** (5‑fold stratified CV):

  - `pca__n_components`: [20, 50, 100, 200]  
  - `knn__n_neighbors`: [3, 5, 7]

**Best configuration**

- `pca__n_components = 20`  
- `knn__n_neighbors = 3`  
- Best cross‑validated accuracy ≈ **0.715** on the training folds.

**Final test performance**

- Test accuracy with best PCA + k‑NN ≈ **0.742**.

**Interpretation**

- Cross‑validation preferred a **more compressed** representation (20 components instead of 50) plus a **smaller neighbourhood** (k=3).
- This combination balanced:
  - enough information to distinguish classes, and  
  - enough noise reduction to make distances meaningful.

## 5. Key Lessons

- **Dimensionality has a big impact.**  
  k‑NN on raw 4096‑dimensional pixels is weak; the same model in a 20–50D PCA space is much stronger.

- **PCA is a learned sketch space.**  
  It captures dominant strokes/structures; k‑NN works better when distance is measured in this simpler space.

- **Cross-validation matters.**  
  Instead of guessing PCA dimension and k, GridSearchCV used data to find a better sweet spot (PCA‑20, k=3).

- **Chinese MNIST is harder than the small digits dataset.**  
  More classes (15), higher resolution (64×64), and more handwriting variation all make the problem tougher, so accuracies are naturally lower even with good models.

Overall, this notebook shows how combining **PCA for dimensionality reduction**, **k‑NN for simple neighbour-based classification**, and **cross‑validation for tuning** can lift performance from about **0.43** to around **0.74** on a challenging image dataset.

In [1]:
import pandas as pd
import numpy as np

csv_path = "/kaggle/input/datasets/fedesoriano/chinese-mnist-digit-recognizer/chineseMNIST.csv"

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("First 10 columns:", df.columns[:10].tolist())
df.head()


Shape: (15000, 4098)
First 10 columns: ['pixel_0', 'pixel_1', 'pixel_2', 'pixel_3', 'pixel_4', 'pixel_5', 'pixel_6', 'pixel_7', 'pixel_8', 'pixel_9']


,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,...,pixel_4088,pixel_4089,pixel_4090,pixel_4091,pixel_4092,pixel_4093,pixel_4094,pixel_4095,label,character
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,九
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,九
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,九
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,九
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,九


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

csv_path = "/kaggle/input/datasets/fedesoriano/chinese-mnist-digit-recognizer/chineseMNIST.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)

# 1) Separate pixel columns and label
pixel_cols = [c for c in df.columns if c.startswith("pixel_")]
print("Number of pixel columns:", len(pixel_cols))

X = df[pixel_cols].values          # image data
y = df["label"].values             # digit label (0–9)
chars = df["character"].values     # Chinese character, if you ever need it

# 2) Infer image side length (should be 64)
side = int(np.sqrt(len(pixel_cols)))
print("Inferred image side:", side)

# 3) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# 4) k-NN classifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("k-NN accuracy on Chinese MNIST:", acc)


Shape: (15000, 4098)
Number of pixel columns: 4096
Inferred image side: 64
k-NN accuracy on Chinese MNIST: 0.43


In [3]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# X, y as before
pca = PCA(n_components=50, random_state=0)
X_pca = pca.fit_transform(X)

print("X_pca shape:", X_pca.shape)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pca, y, test_size=0.2, random_state=0, stratify=y
)

knn_p = KNeighborsClassifier(n_neighbors=5)
knn_p.fit(X_train_p, y_train_p)

y_pred_p = knn_p.predict(X_test_p)
acc_p = accuracy_score(y_test_p, y_pred_p)
print("k-NN accuracy on PCA-50 features:", acc_p)


X_pca shape: (15000, 50)
k-NN accuracy on PCA-50 features: 0.7086666666666667


PCA + k‑NN + cross‑validation

In [4]:
from sklearn.model_selection import train_test_split

# X, y from the Chinese MNIST CSV (4096 pixels, 15 labels)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


In [5]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score

# Pipeline: first PCA, then k-NN
pipe = Pipeline([
    ("pca", PCA(random_state=0)),
    ("knn", KNeighborsClassifier())
])

# Hyperparameters to search
param_grid = {
    "pca__n_components": [20, 50, 100, 200],
    "knn__n_neighbors": [3, 5, 7]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters: {'knn__n_neighbors': 3, 'pca__n_components': 20}
Best CV accuracy: 0.7151666666666666


In [6]:
best_model = grid.best_estimator_
y_pred_test = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
print("Test accuracy with best PCA + k-NN:", test_acc)


Test accuracy with best PCA + k-NN: 0.7423333333333333
